例えば、大学前のアパートを入力とし、そこから20分で移動できる場所を取得したい。以下の合計を求めることになる
・入力から徒歩20分のエリア
・入力から20分以内に到達可能な交通機関から(20分-駅までの時間)分で到着できる別の駅とそこから(20分-(駅までの時間+別の駅までの乗車時間))分以内で到達できるエリア、再帰の可能性あり
手順
1. 20分以内の徒歩エリアを求める
2. 20分以内で到達できる徒歩での乗り換えが必要ない交通機関を求める(以下ダイレクトと呼称)
3. (ダイレクトまでの徒歩時間と、ダイレクトまでの乗車時間を引いた)時間以内の、ダイレクトからの徒歩エリアを取得
4. 2, 3を繰り返す.
5. すべてを結合

2のため、自由位置入力から最寄りの交通機関の検索を作成する

In [6]:
from dotenv import load_dotenv
from engine.mapbox import MapBoxApi, IsochroneProfile
import os
from typing import Optional
from engine import Station, TransitType, get_route_yahoo_transit
from engine import Geometry, Coordinate
from engine.bus import *
from engine.train import *
import geopandas
from shapely.geometry import Point

# データの読み込み
dataset: dict[TransitType, list[Station]] = {
    TransitType.BUS: load_stop_data("../dataset/busstops/kanagawa/P11-22_14.geojson"),
    TransitType.TRAIN: load_station_data("../dataset/stations/N02-20_Station.geojson"),
}
load_dotenv()
mapbox_api = MapBoxApi(os.getenv("MAPBOX_API_TOKEN"))

In [7]:
# 神奈川工科大学を入力とする
user_input = Geometry(Type="?", Coordinates=[Coordinate(Lat=35.4864695, Lng=139.3417615)])

## バス停の検索

### 入力からのIsochroneで検索

In [8]:
# バス停までは徒歩とする. 10~30分以内でたどり着ける最寄りのバス停をリストアップする
time_limits = [10, 20, 30]
nearby_bus_stops: dict[int, list[Station]] = {
    10: [],
    20: [],
    30: [],
}
walking_areas_with_time_limit: dict[int, Optional[geopandas.GeoDataFrame]] = {
    10: None,
    20: None,
    30: None,
}

# 制限時間を用いて, isochroneを取得
isochrones: dict = mapbox_api.get_isochrone(
        prof = IsochroneProfile.Walking,   # 移動方法
        coordinate=user_input.calc_mean(), # 基準
        contours_minutes=time_limits,      # 所要時間ズ
)

# 制限時間ごとに取り出して格納
for time_limit in time_limits:
    # timelimit==contourを取り出す
    isochrone: dict = [isochrone for isochrone in isochrones["features"] if isochrone["properties"]["contour"] == time_limit][0] 
    # 上でFeaturesからFeatureを取り出した. これ単体でもう一度collectionをつくってあげる
    isochrone_gpd = geopandas.GeoDataFrame.from_features(
        {"features": [isochrone], "type": "FeatureCollection"}
    )
    walking_areas_with_time_limit[time_limit] = isochrone_gpd

# バス停が各制限時間エリアに含まれるかチェック
for bus_stop in dataset[TransitType.BUS]:
    # Geopandasのために座標をPoint化
    bus_stop_coordinate = bus_stop.geometry.calc_mean()
    bus_stop_point = Point(bus_stop_coordinate.Lng, bus_stop_coordinate.Lat)
    for time_limit, walking_area in walking_areas_with_time_limit.items():
        # このバス停の平均座標は徒歩x分のエリアに含まれますか?
        is_include = walking_area.contains(bus_stop_point)[0]
        if is_include:
            # 含まれる場合は記録
            nearby_bus_stops[time_limit].append(bus_stop)

# print(walking_areas_with_time_limit[10].contains(Point(user_input.calc_mean().Lng, user_input.calc_mean().Lat)))

# 最終表示
for time_limit, bus_stops in nearby_bus_stops.items():
    print(f"徒歩{time_limit}分以内の最寄りのバス停")
    for bus_stop in bus_stops:
        print(f"  - {bus_stop.name}")


徒歩10分以内の最寄りのバス停
  - 下谷
  - 上三田
  - 西四ツ谷
  - リコー前
  - 十軒村
  - 神奈川工科大学
  - 神奈川工科大学前
  - 子中
徒歩20分以内の最寄りのバス停
  - 下川入
  - 才戸橋
  - 棚沢
  - 鳶尾一丁目
  - 鳶尾小学校入口
  - 下谷
  - 上三田
  - 西四ツ谷
  - 鳶尾団地東
  - リコー前
  - 十軒村
  - 神奈川工科大学
  - 神奈川工科大学前
  - 公所
  - 新道公所
  - 桝割
  - 睦合北公民館前
  - 子中
  - 子合
  - 山中陣屋跡公園前
  - 清源院前
  - 荻野新宿
  - 山王坂下
  - 山王坂上
  - 宿原入口
  - 小山
  - 糀屋前
徒歩30分以内の最寄りのバス停
  - 坂本
  - 坂本
  - 下川入
  - 才戸橋
  - あつぎ郷土博物館
  - 棚沢
  - 鳶尾一丁目
  - 鳶尾団地
  - 鳶尾小学校入口
  - 下谷
  - 上三田
  - 荻野神社入口
  - 西四ツ谷
  - 鳶尾四丁目
  - 鳶尾団地東
  - 鳶尾山前
  - リコー前
  - 十軒村
  - 稲荷木
  - 神奈川工科大学
  - うとう坂
  - 神奈川工科大学前
  - 公所
  - 新道公所
  - 桝割
  - 睦合北公民館前
  - 子中
  - 子合
  - 山中陣屋跡公園前
  - 清源院前
  - 荻野新宿
  - 山王坂下
  - 東河原入口
  - 屋際
  - 山王坂上
  - 宿原入口
  - 三田十日市場
  - 小山
  - 糀屋前
  - 天台上野原
  - 松蓮寺
  - 及川富士塚
  - 及川球技場入口
  - 千頭
  - 及川
  - 千頭坂下


In [12]:
def get_nearby_stations(
        dataset: dict[TransitType, list[Station]],
        user_input: Coordinate,
        time_limits: list[int],
        transit_types: list[TransitType]
) -> dict[int, list[Station]]:
    nearby_stations: dict[int, list[Station]] = {tl: [] for tl in time_limits}
    walking_areas_with_time_limit: dict[int, geopandas.GeoDataFrame] = {}

    # MapboxApiを用いてisochroneを取得
    isochrones: dict = mapbox_api.get_isochrone(
        prof=IsochroneProfile.Walking,
        coordinate=user_input,
        contours_minutes=time_limits,
    )

    # Isochroneの加工
    for time_limit in time_limits:
        isochrone: dict = [isochrone for isochrone in isochrones["features"] if isochrone["properties"]["contour"] == time_limit][0]
        isochrone_gpd = geopandas.GeoDataFrame.from_features(
            {"features": [isochrone], "type": "FeatureCollection"}
        )
        walking_areas_with_time_limit[time_limit] = isochrone_gpd


    # チェック対象を確認
    candidates: list[Station] = []
    if TransitType.BUS in transit_types:
        candidates += dataset[TransitType.BUS]
    if TransitType.TRAIN in transit_types:
        candidates += dataset[TransitType.TRAIN]

    # チェック
    for candidate in candidates:
        candidate_coord = candidate.geometry.calc_mean()
        candidate_point = Point(candidate_coord.Lng, candidate_coord.Lat)
        for time_limit, walking_area in walking_areas_with_time_limit.items():
            is_include = bool(walking_area.contains(candidate_point)[0])
            if is_include:
                nearby_stations[time_limit].append(candidate)

    return nearby_stations

In [15]:
print("is same output:", nearby_bus_stops == get_nearby_stations(dataset, user_input.calc_mean(), time_limits, [TransitType.BUS]))

# for time_limit, stations in get_nearby_stations(dataset, user_input.calc_mean(), time_limits, [TransitType.BUS]).items():
#     print(f"徒歩{time_limit}分以内の最寄りのバス停/駅")
#     for station in stations:
#         print(f"  - {station.name}")

is same output: True


In [21]:
# 厚木市役所
user_input = Geometry(Type="?", Coordinates=[Coordinate(Lat=35.4429973, Lng=139.3611488,)])
for time_limit, stations in get_nearby_stations(dataset, user_input.calc_mean(), time_limits, [TransitType.BUS, TransitType.TRAIN]).items():
    print(f"徒歩{time_limit}分以内の最寄りのバス停/駅")
    for station in stations:
        print(f"  - {station.management_groups[0]} {station.name}")

徒歩10分以内の最寄りのバス停/駅
  - 神奈川中央交通（株） 市立病院前
  - 神奈川中央交通（株） 松枝町一丁目
  - 神奈川中央交通（株） 税務署入口
  - 神奈川中央交通（株） 厚木警察署前
  - 神奈川中央交通（株） 合同庁舎前
  - 神奈川中央交通（株） 中央通り
  - 神奈川中央交通（株） 小田急通り
  - 神奈川中央交通（株） 市役所入口
  - 神奈川中央交通（株） あつぎ大通り
  - 神奈川中央交通（株） 厚木バスセンター
  - 神奈川中央交通（株） 本厚木駅
  - 神奈川中央交通西（株） 本厚木駅北口
  - 神奈川中央交通（株） 栄町二丁目
  - 神奈川中央交通（株） 本厚木駅南口
  - 神奈川中央交通（株） 中町四丁目
  - 小田急電鉄 本厚木
徒歩20分以内の最寄りのバス停/駅
  - 神奈川中央交通（株） 三家入口
  - 神奈川中央交通（株） 吾妻団地
  - 神奈川中央交通（株） 木売場
  - 神奈川中央交通（株） 鮎津橋
  - 神奈川中央交通（株） 愛光病院前
  - 神奈川中央交通（株） 戸室神社下
  - 神奈川中央交通（株） 市立病院前
  - 神奈川中央交通（株） 松枝町一丁目
  - 神奈川中央交通（株） 戸室住宅前
  - 神奈川中央交通（株） 元町
  - 神奈川中央交通（株） 税務署入口
  - 神奈川中央交通（株） 厚木警察署前
  - 神奈川中央交通（株） 日立アステモ前
  - 神奈川中央交通（株） 厚木東町
  - 神奈川中央交通（株） 厚木高校前
  - 神奈川中央交通（株） 合同庁舎前
  - 神奈川中央交通（株） 日立アステモ前
  - 神奈川中央交通（株） 中央通り
  - 神奈川中央交通（株） 小田急通り
  - 神奈川中央交通（株） 市役所入口
  - 神奈川中央交通（株） 天王町
  - 神奈川中央交通（株） あつぎ大通り
  - 神奈川中央交通（株） 片岸
  - 神奈川中央交通（株） 厚木バスセンター
  - 神奈川中央交通（株） 仲町
  - 神奈川中央交通（株） 本厚木駅
  - 神奈川中央交通西（株） 本厚木駅北口
  - 神奈川中央交通（株） 栄町二丁目
  - 神奈川中央交通（株） 本厚木駅南口
  - 神奈川中央交通（株） 恩名公民館前
  - 